In [1]:
!pip install -q transformers accelerate sentencepiece hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os, json, shutil, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from tqdm import tqdm
from pathlib import Path

# ---- Paths (adjust if your dataset slugs differ) ----
DATASET_ROOT  = '/kaggle/input/datasets/varun000reddy/deepfashion-yolocrops'
CROP_DATA_DIR = f'{DATASET_ROOT}/cropped_images'
META_CSV      = f'{DATASET_ROOT}/crop_metadata.csv'

WORK_DIR      = '/kaggle/working'
CAPTIONS_FILE = f'{WORK_DIR}/captions_full.parquet'
MERGED_CSV    = f'{WORK_DIR}/crop_metadata_with_captions.csv'
IMG_EMB_FILE  = f'{WORK_DIR}/img_embeddings.npy'
TXT_EMB_FILE  = f'{WORK_DIR}/txt_embeddings.npy'
META_PARQUET  = f'{WORK_DIR}/all_meta_ordered.parquet'
FT_CKPT       = f'{WORK_DIR}/ft_clip_final.pt'
FT_EMB_FILE   = f'{WORK_DIR}/ft_img_embeddings.npy'
RESULTS_FILE  = f'{WORK_DIR}/results_all_ablations.json'

# Devices
print(f"CUDA devices: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

DEVICE_BLIP = 'cuda:0'
DEVICE_CLIP = 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0'
print(f"BLIP-2 on {DEVICE_BLIP}, CLIP on {DEVICE_CLIP}")

CUDA devices: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
BLIP-2 on cuda:0, CLIP on cuda:1


In [3]:
all_meta = pd.read_csv(META_CSV)
print(f"Loaded {len(all_meta)} rows")

OLD_PREFIX = '/kaggle/working/cropped/'
NEW_PREFIX = f'{CROP_DATA_DIR}/'
all_meta['crop_path'] = all_meta['crop_path'].str.replace(OLD_PREFIX, NEW_PREFIX, regex=False)

sample_new = all_meta['crop_path'].iloc[0]
print(f"Sample path: {sample_new}")
print(f"Exists: {os.path.exists(sample_new)}")

exists_mask = all_meta['crop_path'].apply(os.path.exists)
print(f"Valid crop paths: {exists_mask.sum()}/{len(all_meta)}")
all_meta = all_meta[exists_mask].reset_index(drop=True)

print(f"\nSplit counts:\n{all_meta.split.value_counts()}")

# Hard assert — don't waste hours on broken paths
assert len(all_meta) > 1000, "Too few valid paths! Fix prefix before running."

Loaded 52712 rows
Sample path: /kaggle/input/datasets/varun000reddy/deepfashion-yolocrops/cropped_images/gallery/WOMEN/Blouses_Shirts/id_00000001/02_1_front.jpg
Exists: True
Valid crop paths: 52712/52712

Split counts:
split
train      25882
query      14218
gallery    12612
Name: count, dtype: int64


In [4]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration

BLIP2_MODEL = "Salesforce/blip2-opt-2.7b"
print(f"Loading BLIP-2 on {DEVICE_BLIP}...")
blip2_processor = Blip2Processor.from_pretrained(BLIP2_MODEL)
blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    BLIP2_MODEL, torch_dtype=torch.float16,
).to(DEVICE_BLIP).eval()
print(f"BLIP-2 loaded. VRAM on GPU 0: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

Loading BLIP-2 on cuda:0...


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

BLIP-2 loaded. VRAM on GPU 0: 7.72 GB


In [5]:
BLIP_PROMPT = "Question: what clothing item is shown in this image? Answer:"
BLIP_BATCH_SIZE = 8


@torch.no_grad()
def blip2_caption_batch(pil_images, prompt=BLIP_PROMPT):
    prompts = [prompt] * len(pil_images)
    inputs = blip2_processor(
        images=pil_images, text=prompts, return_tensors="pt", padding=True,
    ).to(DEVICE_BLIP, torch.float16)
    out = blip2_model.generate(**inputs, max_new_tokens=40, do_sample=False, num_beams=1)
    captions = []
    for o in out:
        text = blip2_processor.decode(o, skip_special_tokens=True).strip()
        if prompt in text:
            text = text.split(prompt)[-1].strip()
        text = text.split('\n')[0].strip()
        captions.append(text)
    return captions


def caption_all(meta_df, batch_size=BLIP_BATCH_SIZE, checkpoint_every=100):
    if os.path.exists(CAPTIONS_FILE):
        cached = pd.read_parquet(CAPTIONS_FILE)
        print(f"Loaded {len(cached)} cached captions")
    else:
        cached = pd.DataFrame(columns=['crop_path', 'caption'])
    
    done = set(cached.crop_path)
    todo = meta_df[~meta_df.crop_path.isin(done)].reset_index(drop=True)
    print(f"To caption: {len(todo)} new images")
    if len(todo) == 0:
        return cached
    
    new_rows = []
    for start in tqdm(range(0, len(todo), batch_size), desc="BLIP-2"):
        batch_rows = todo.iloc[start:start + batch_size]
        try:
            imgs = [Image.open(p).convert("RGB") for p in batch_rows['crop_path']]
            caps = blip2_caption_batch(imgs)
        except Exception:
            caps = []
            for p in batch_rows['crop_path']:
                try:
                    img = Image.open(p).convert("RGB")
                    caps.append(blip2_caption_batch([img])[0])
                except Exception:
                    caps.append('')
        
        for p, c in zip(batch_rows['crop_path'], caps):
            new_rows.append({'crop_path': p, 'caption': c})
        
        batch_num = start // batch_size + 1
        if batch_num % checkpoint_every == 0:
            tmp = pd.concat([cached, pd.DataFrame(new_rows)], ignore_index=True)
            tmp.to_parquet(CAPTIONS_FILE)
    
    cached = pd.concat([cached, pd.DataFrame(new_rows)], ignore_index=True)
    cached.to_parquet(CAPTIONS_FILE)
    print(f"Saved {len(cached)} captions to {CAPTIONS_FILE}")
    return cached

In [6]:
print(f"\n{'='*60}\nPHASE 2: BLIP-2 CAPTIONING\n{'='*60}")
t0 = time.time()

captions_df = caption_all(all_meta, batch_size=BLIP_BATCH_SIZE)

all_meta = all_meta.drop(columns=['caption'], errors='ignore').merge(
    captions_df, on='crop_path', how='left'
)
all_meta['caption'] = all_meta['caption'].fillna('a clothing item')
all_meta.to_csv(MERGED_CSV, index=False)

print(f"\nPhase 2 done in {(time.time()-t0)/60:.1f} min")
print(f"Sample captions:")
print(all_meta[['split','category','caption']].sample(8, random_state=0).to_string(index=False))


PHASE 2: BLIP-2 CAPTIONING
To caption: 52712 new images


BLIP-2: 100%|██████████| 6589/6589 [1:24:15<00:00,  1.30it/s]


Saved 52712 captions to /kaggle/working/captions_full.parquet

Phase 2 done in 84.3 min
Sample captions:
  split   category                            caption
gallery    Dresses                   a burgundy dress
  query      Pants the blue and white patterned pants
  train Tees_Tanks                           mesh top
  train Tees_Tanks             black asymmetric dress
  train   Sweaters                         camel coat
  train   Leggings                 black skinny jeans
gallery Tees_Tanks                    striped t-shirt
  train      Pants                         camo pants


In [7]:
del blip2_model, blip2_processor
torch.cuda.empty_cache()
print("BLIP-2 unloaded.")
print(f"GPU 0 VRAM: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

BLIP-2 unloaded.
GPU 0 VRAM: 0.01 GB


In [8]:
from transformers import CLIPProcessor, CLIPModel

CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE_CLIP).eval()
EMBED_DIM = clip_model.config.projection_dim
print(f"CLIP loaded on {DEVICE_CLIP}. Dim={EMBED_DIM}")

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP loaded on cuda:1. Dim=512


In [9]:
def _to_tensor(x):
    if isinstance(x, torch.Tensor):
        return x
    for attr in ('image_embeds','text_embeds','pooler_output','last_hidden_state'):
        if hasattr(x, attr):
            t = getattr(x, attr)
            if isinstance(t, torch.Tensor):
                return t
    raise TypeError(f"Cannot unwrap {type(x).__name__}")


@torch.no_grad()
def clip_image_embed_batch(pil_images, model=None):
    if model is None: model = clip_model
    inputs = clip_processor(images=pil_images, return_tensors="pt").to(DEVICE_CLIP)
    feats = model.get_image_features(pixel_values=inputs['pixel_values'])
    feats = _to_tensor(feats)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy()


@torch.no_grad()
def clip_text_embed_batch(texts, model=None):
    if model is None: model = clip_model
    inputs = clip_processor(text=texts, return_tensors="pt", padding=True, truncation=True).to(DEVICE_CLIP)
    feats = model.get_text_features(
        input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask']
    )
    feats = _to_tensor(feats)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy()

In [10]:
print(f"\n{'='*60}\nPHASE 3: CLIP EMBEDDINGS\n{'='*60}")
t0 = time.time()

def build_embeddings(meta_df, batch_size=64):
    n = len(meta_df)
    img_emb = np.zeros((n, EMBED_DIM), dtype=np.float32)
    txt_emb = np.zeros((n, EMBED_DIM), dtype=np.float32)
    for start in tqdm(range(0, n, batch_size), desc="CLIP embed"):
        batch = meta_df.iloc[start:start + batch_size]
        imgs = [Image.open(p).convert("RGB") for p in batch['crop_path']]
        caps = batch['caption'].fillna('a clothing item').tolist()
        img_emb[start:start + len(batch)] = clip_image_embed_batch(imgs)
        txt_emb[start:start + len(batch)] = clip_text_embed_batch(caps)
    return img_emb, txt_emb


if os.path.exists(IMG_EMB_FILE) and os.path.exists(TXT_EMB_FILE):
    img_embeddings = np.load(IMG_EMB_FILE)
    txt_embeddings = np.load(TXT_EMB_FILE)
    print(f"Loaded cached: img={img_embeddings.shape}, txt={txt_embeddings.shape}")
else:
    img_embeddings, txt_embeddings = build_embeddings(all_meta)
    np.save(IMG_EMB_FILE, img_embeddings)
    np.save(TXT_EMB_FILE, txt_embeddings)

all_meta.to_parquet(META_PARQUET)
print(f"Phase 3 (embeddings) done in {(time.time()-t0)/60:.1f} min")


PHASE 3: CLIP EMBEDDINGS


CLIP embed: 100%|██████████| 824/824 [06:10<00:00,  2.22it/s]


Phase 3 (embeddings) done in 6.2 min


In [11]:
import hnswlib

def build_hnsw_for(gallery_emb):
    n, dim = gallery_emb.shape
    idx = hnswlib.Index(space='cosine', dim=dim)
    idx.init_index(max_elements=n, ef_construction=200, M=16)
    idx.add_items(gallery_emb, ids=np.arange(n))
    idx.set_ef(50)
    return idx


def fuse(img_emb, txt_emb, alpha):
    v = alpha * img_emb + (1 - alpha) * txt_emb
    v = v / np.linalg.norm(v, axis=1, keepdims=True)
    return v.astype(np.float32)


def evaluate_retrieval(query_emb, query_item_ids, gallery_emb, gallery_item_ids, k_values=(5, 10, 15)):
    idx = build_hnsw_for(gallery_emb)
    max_k = max(k_values)
    ids, _ = idx.knn_query(query_emb, k=max_k)
    
    metrics = {}
    for k in k_values:
        recalls, ndcgs, aps = [], [], []
        for i in range(len(query_emb)):
            true_id = query_item_ids[i]
            retrieved = gallery_item_ids[ids[i][:k]]
            rel = (retrieved == true_id).astype(int)
            
            recalls.append(1.0 if rel.sum() > 0 else 0.0)
            
            dcg = np.sum(rel / np.log2(np.arange(2, k + 2)))
            ideal = np.sort(rel)[::-1]
            idcg = np.sum(ideal / np.log2(np.arange(2, k + 2)))
            ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
            
            if rel.sum() > 0:
                cum_hits = np.cumsum(rel)
                precision = cum_hits / np.arange(1, k + 1)
                aps.append(np.sum(precision * rel) / rel.sum())
            else:
                aps.append(0.0)
        
        metrics[f'Recall@{k}'] = float(np.mean(recalls))
        metrics[f'NDCG@{k}']   = float(np.mean(ndcgs))
        metrics[f'mAP@{k}']    = float(np.mean(aps))
    return metrics

In [12]:
print(f"\n{'='*60}\nABLATION A & B (frozen CLIP)\n{'='*60}")

query_mask   = (all_meta.split == 'query').values
gallery_mask = (all_meta.split == 'gallery').values

q_img = img_embeddings[query_mask]
g_img = img_embeddings[gallery_mask]
q_txt = txt_embeddings[query_mask]
g_txt = txt_embeddings[gallery_mask]
q_ids = all_meta.loc[query_mask, 'item_id'].values
g_ids = all_meta.loc[gallery_mask, 'item_id'].values

print(f"Query: {len(q_img)}, Gallery: {len(g_img)}")
print(f"Eligible (true item in gallery): {np.isin(q_ids, g_ids).sum()}")

results = {}

# Ablation A: α=1 (vision only)
print("\n--- Ablation A: vision-only (α=1) ---")
results['A_alpha_1.0'] = evaluate_retrieval(q_img, q_ids, g_img, g_ids)
for k, v in results['A_alpha_1.0'].items():
    print(f"  {k}: {v:.4f}")

# Ablation B: α=0.5 and α=0.7
for alpha in [0.5, 0.7]:
    print(f"\n--- Ablation B: frozen + caption (α={alpha}) ---")
    qf = fuse(q_img, q_txt, alpha)
    gf = fuse(g_img, g_txt, alpha)
    results[f'B_alpha_{alpha}'] = evaluate_retrieval(qf, q_ids, gf, g_ids)
    for k, v in results[f'B_alpha_{alpha}'].items():
        print(f"  {k}: {v:.4f}")

with open(RESULTS_FILE, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nSaved interim results to {RESULTS_FILE}")


ABLATION A & B (frozen CLIP)
Query: 14218, Gallery: 12612
Eligible (true item in gallery): 14218

--- Ablation A: vision-only (α=1) ---
  Recall@5: 0.4015
  NDCG@5: 0.3213
  mAP@5: 0.2907
  Recall@10: 0.4759
  NDCG@10: 0.3403
  mAP@10: 0.2882
  Recall@15: 0.5200
  NDCG@15: 0.3483
  mAP@15: 0.2827

--- Ablation B: frozen + caption (α=0.5) ---
  Recall@5: 0.4980
  NDCG@5: 0.3998
  mAP@5: 0.3617
  Recall@10: 0.5825
  NDCG@10: 0.4203
  mAP@10: 0.3558
  Recall@15: 0.6324
  NDCG@15: 0.4282
  mAP@15: 0.3469

--- Ablation B: frozen + caption (α=0.7) ---
  Recall@5: 0.5167
  NDCG@5: 0.4173
  mAP@5: 0.3784
  Recall@10: 0.6035
  NDCG@10: 0.4377
  mAP@10: 0.3713
  Recall@15: 0.6536
  NDCG@15: 0.4452
  mAP@15: 0.3615

Saved interim results to /kaggle/working/results_all_ablations.json


In [13]:
print(f"\n{'='*60}\nPHASE 4: CLIP FINE-TUNING\n{'='*60}")

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])


class InShopContrastiveDataset(Dataset):
    def __init__(self, train_meta, transform):
        self.transform = transform
        grouped = train_meta[train_meta.crop_path.notna()].groupby('item_id')['crop_path'].apply(list)
        self.items = grouped[grouped.apply(len) >= 2].to_dict()
        self.item_ids = list(self.items.keys())
        print(f"Contrastive dataset: {len(self.item_ids)} items with >=2 views")
    
    def __len__(self):
        return len(self.item_ids)
    
    def __getitem__(self, idx):
        item_id = self.item_ids[idx]
        views = self.items[item_id]
        v1, v2 = random.sample(views, 2)
        img1 = Image.open(v1).convert("RGB")
        img2 = Image.open(v2).convert("RGB")
        return self.transform(img1), self.transform(img2)


train_meta = all_meta[all_meta.split == 'train'].copy()
train_dataset = InShopContrastiveDataset(train_meta, train_transform)
train_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True,
)
print(f"Batches per epoch: {len(train_loader)}")


PHASE 4: CLIP FINE-TUNING
Contrastive dataset: 3985 items with >=2 views
Batches per epoch: 62


In [14]:
# Use the same CLIP model — just unfreeze the right pieces
for p in clip_model.parameters():
    p.requires_grad = False

for block in clip_model.vision_model.encoder.layers[-4:]:
    for p in block.parameters():
        p.requires_grad = True
for p in clip_model.visual_projection.parameters():
    p.requires_grad = True
for p in clip_model.vision_model.post_layernorm.parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in clip_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in clip_model.parameters())
print(f"Trainable: {trainable:,}/{total:,} ({100*trainable/total:.1f}%)")

Trainable: 28,746,240/151,277,313 (19.0%)


In [15]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 5
LR = 1e-5
TEMPERATURE = 0.07

def info_nce_loss(emb1, emb2, temperature=TEMPERATURE):
    N = emb1.shape[0]
    logits = emb1 @ emb2.T / temperature
    labels = torch.arange(N, device=logits.device)
    return (nn.functional.cross_entropy(logits, labels) +
            nn.functional.cross_entropy(logits.T, labels)) / 2


trainable_params = [p for p in clip_model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))

t0 = time.time()
history = []

for epoch in range(EPOCHS):
    clip_model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for img1, img2 in pbar:
        img1 = img1.to(DEVICE_CLIP, non_blocking=True)
        img2 = img2.to(DEVICE_CLIP, non_blocking=True)
        
        e1 = clip_model.get_image_features(pixel_values=img1)
        e2 = clip_model.get_image_features(pixel_values=img2)
        e1 = _to_tensor(e1); e2 = _to_tensor(e2)
        e1 = e1 / e1.norm(dim=-1, keepdim=True)
        e2 = e2 / e2.norm(dim=-1, keepdim=True)
        
        loss = info_nce_loss(e1, e2)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        epoch_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")
    
    avg = epoch_loss / len(train_loader)
    history.append(avg)
    print(f"Epoch {epoch+1}: avg loss = {avg:.4f}")
    
    # Save checkpoint every epoch (so a crash doesn't lose everything)
    torch.save(clip_model.state_dict(), f'{WORK_DIR}/ft_clip_epoch{epoch+1}.pt')

torch.save(clip_model.state_dict(), FT_CKPT)
print(f"\nFine-tuning done in {(time.time()-t0)/60:.1f} min")
print(f"Loss history: {[f'{h:.4f}' for h in history]}")

Epoch 1/5: 100%|██████████| 62/62 [00:42<00:00,  1.47it/s, loss=0.5923, lr=9.05e-06]


Epoch 1: avg loss = 1.1013


Epoch 2/5: 100%|██████████| 62/62 [00:40<00:00,  1.54it/s, loss=0.4149, lr=6.55e-06]


Epoch 2: avg loss = 0.6158


Epoch 3/5: 100%|██████████| 62/62 [00:40<00:00,  1.54it/s, loss=0.4020, lr=3.45e-06]


Epoch 3: avg loss = 0.5128


Epoch 4/5: 100%|██████████| 62/62 [00:40<00:00,  1.54it/s, loss=0.5698, lr=9.55e-07]


Epoch 4: avg loss = 0.4694


Epoch 5/5: 100%|██████████| 62/62 [00:41<00:00,  1.49it/s, loss=0.3415, lr=0.00e+00]


Epoch 5: avg loss = 0.4559

Fine-tuning done in 3.5 min
Loss history: ['1.1013', '0.6158', '0.5128', '0.4694', '0.4559']


In [16]:
print(f"\n{'='*60}\nPHASE 5: ABLATION C (fine-tuned)\n{'='*60}")
clip_model.eval()
t0 = time.time()

n = len(all_meta)
ft_img_emb = np.zeros((n, EMBED_DIM), dtype=np.float32)

if os.path.exists(FT_EMB_FILE):
    ft_img_emb = np.load(FT_EMB_FILE)
    print(f"Loaded cached fine-tuned embeddings: {ft_img_emb.shape}")
else:
    for start in tqdm(range(0, n, 64), desc="FT CLIP embed"):
        batch = all_meta.iloc[start:start + 64]
        imgs = [Image.open(p).convert("RGB") for p in batch['crop_path']]
        ft_img_emb[start:start + len(batch)] = clip_image_embed_batch(imgs)
    np.save(FT_EMB_FILE, ft_img_emb)
    print(f"Saved fine-tuned embeddings in {(time.time()-t0)/60:.1f} min")


PHASE 5: ABLATION C (fine-tuned)


FT CLIP embed: 100%|██████████| 824/824 [06:06<00:00,  2.25it/s]

Saved fine-tuned embeddings in 6.1 min


In [17]:
q_ft_img = ft_img_emb[query_mask]
g_ft_img = ft_img_emb[gallery_mask]

# C with vision only (fine-tuned)
print("\n--- Ablation C: fine-tuned CLIP, vision-only (α=1) ---")
results['C_alpha_1.0'] = evaluate_retrieval(q_ft_img, q_ids, g_ft_img, g_ids)
for k, v in results['C_alpha_1.0'].items():
    print(f"  {k}: {v:.4f}")

# C with text fusion at α=0.7
for alpha in [0.7]:
    print(f"\n--- Ablation C: fine-tuned + caption (α={alpha}) ---")
    qf = fuse(q_ft_img, q_txt, alpha)
    gf = fuse(g_ft_img, g_txt, alpha)
    results[f'C_alpha_{alpha}'] = evaluate_retrieval(qf, q_ids, gf, g_ids)
    for k, v in results[f'C_alpha_{alpha}'].items():
        print(f"  {k}: {v:.4f}")

with open(RESULTS_FILE, 'w') as f:
    json.dump(results, f, indent=2)


--- Ablation C: fine-tuned CLIP, vision-only (α=1) ---
  Recall@5: 0.7558
  NDCG@5: 0.6466
  mAP@5: 0.5987
  Recall@10: 0.8245
  NDCG@10: 0.6551
  mAP@10: 0.5748
  Recall@15: 0.8578
  NDCG@15: 0.6547
  mAP@15: 0.5556

--- Ablation C: fine-tuned + caption (α=0.7) ---
  Recall@5: 0.7826
  NDCG@5: 0.6773
  mAP@5: 0.6308
  Recall@10: 0.8435
  NDCG@10: 0.6830
  mAP@10: 0.6057
  Recall@15: 0.8732
  NDCG@15: 0.6813
  mAP@15: 0.5858


In [18]:
print(f"\n{'='*60}\nFINAL RESULTS SUMMARY\n{'='*60}\n")

df = pd.DataFrame(results).T
df = df.round(4)
print(df.to_string())

df.to_csv(f'{WORK_DIR}/final_results.csv')
print(f"\nSaved to {WORK_DIR}/final_results.csv")
print(f"All artifacts in {WORK_DIR}:")
!ls -lh {WORK_DIR} | head -30


FINAL RESULTS SUMMARY

             Recall@5  NDCG@5   mAP@5  Recall@10  NDCG@10  mAP@10  Recall@15  NDCG@15  mAP@15
A_alpha_1.0    0.4015  0.3213  0.2907     0.4759   0.3403  0.2882     0.5200   0.3483  0.2827
B_alpha_0.5    0.4980  0.3998  0.3617     0.5825   0.4203  0.3558     0.6324   0.4282  0.3469
B_alpha_0.7    0.5167  0.4173  0.3784     0.6035   0.4377  0.3713     0.6536   0.4452  0.3615
C_alpha_1.0    0.7558  0.6466  0.5987     0.8245   0.6551  0.5748     0.8578   0.6547  0.5556
C_alpha_0.7    0.7826  0.6773  0.6308     0.8435   0.6830  0.6057     0.8732   0.6813  0.5858

Saved to /kaggle/working/final_results.csv
All artifacts in /kaggle/working:
total 3.8G
-rw-r--r-- 1 root root 1.5M May 12 18:45 all_meta_ordered.parquet
-rw-r--r-- 1 root root 850K May 12 18:39 captions_full.parquet
-rw-r--r-- 1 root root  14M May 12 18:39 crop_metadata_with_captions.csv
-rw-r--r-- 1 root root  444 May 12 18:55 final_results.csv
-rw-r--r-- 1 root root 578M May 12 18:46 ft_clip_epoch1.pt
-rw

In [19]:
import os, json
import numpy as np
import pandas as pd
import hnswlib

WORK_DIR = '/kaggle/working'

# ---- Reload everything from disk ----
print("Reloading from disk...")
all_meta       = pd.read_parquet(f'{WORK_DIR}/all_meta_ordered.parquet')
img_embeddings = np.load(f'{WORK_DIR}/img_embeddings.npy')
txt_embeddings = np.load(f'{WORK_DIR}/txt_embeddings.npy')
ft_img_emb     = np.load(f'{WORK_DIR}/ft_img_embeddings.npy')

print(f"  all_meta:       {len(all_meta)} rows")
print(f"  img_embeddings: {img_embeddings.shape}")
print(f"  txt_embeddings: {txt_embeddings.shape}")
print(f"  ft_img_emb:     {ft_img_emb.shape}")

# ---- Recompute splits ----
query_mask   = (all_meta.split == 'query').values
gallery_mask = (all_meta.split == 'gallery').values

g_img    = img_embeddings[gallery_mask]
g_txt    = txt_embeddings[gallery_mask]
g_ft_img = ft_img_emb[gallery_mask]
g_item_ids = all_meta.loc[gallery_mask, 'item_id'].values

print(f"\nGallery: {len(g_img)} items")


# ---- Helper functions ----
def fuse(img_emb, txt_emb, alpha):
    v = alpha * img_emb + (1 - alpha) * txt_emb
    v = v / np.linalg.norm(v, axis=1, keepdims=True)
    return v.astype(np.float32)


def save_hnsw(embeddings, item_ids, filename, hnsw_dir):
    n, dim = embeddings.shape
    idx = hnswlib.Index(space='cosine', dim=dim)
    idx.init_index(max_elements=n, ef_construction=200, M=16)
    idx.add_items(embeddings, ids=np.arange(n))
    idx.set_ef(50)
    
    idx_path = os.path.join(hnsw_dir, f'{filename}.bin')
    idx.save_index(idx_path)
    
    meta_path = os.path.join(hnsw_dir, f'{filename}_item_ids.npy')
    np.save(meta_path, item_ids)
    
    size_mb = os.path.getsize(idx_path) / (1024 * 1024)
    print(f"  ✓ {filename}.bin ({size_mb:.1f} MB) + {filename}_item_ids.npy")


# ---- Build all indexes ----
HNSW_DIR = f'{WORK_DIR}/hnsw_indexes'
os.makedirs(HNSW_DIR, exist_ok=True)

print(f"\n{'='*60}\nBUILDING HNSW INDEXES\n{'='*60}\n")

print("Ablation A — vision-only frozen CLIP:")
save_hnsw(g_img, g_item_ids, 'gallery_A_vision_only', HNSW_DIR)

print("\nAblation B — frozen + caption (α=0.7):")
g_fused_B = fuse(g_img, g_txt, alpha=0.7)
save_hnsw(g_fused_B, g_item_ids, 'gallery_B_alpha_0.7', HNSW_DIR)

print("\nAblation C — fine-tuned, vision-only:")
save_hnsw(g_ft_img, g_item_ids, 'gallery_C_finetuned_vision', HNSW_DIR)

print("\nAblation C — fine-tuned + caption (α=0.7) — PRIMARY for demo:")
g_fused_C = fuse(g_ft_img, g_txt, alpha=0.7)
save_hnsw(g_fused_C, g_item_ids, 'gallery_C_alpha_0.7_PRIMARY', HNSW_DIR)

# Save gallery metadata for the demo
gallery_meta = all_meta[gallery_mask].reset_index(drop=True)
gallery_meta.to_parquet(os.path.join(HNSW_DIR, 'gallery_metadata.parquet'))

print(f"\n✓ All indexes saved to {HNSW_DIR}/")
!ls -lh {HNSW_DIR}

Reloading from disk...
  all_meta:       52712 rows
  img_embeddings: (52712, 512)
  txt_embeddings: (52712, 512)
  ft_img_emb:     (52712, 512)

Gallery: 12612 items

BUILDING HNSW INDEXES

Ablation A — vision-only frozen CLIP:
  ✓ gallery_A_vision_only.bin (26.4 MB) + gallery_A_vision_only_item_ids.npy

Ablation B — frozen + caption (α=0.7):
  ✓ gallery_B_alpha_0.7.bin (26.4 MB) + gallery_B_alpha_0.7_item_ids.npy

Ablation C — fine-tuned, vision-only:
  ✓ gallery_C_finetuned_vision.bin (26.4 MB) + gallery_C_finetuned_vision_item_ids.npy

Ablation C — fine-tuned + caption (α=0.7) — PRIMARY for demo:
  ✓ gallery_C_alpha_0.7_PRIMARY.bin (26.4 MB) + gallery_C_alpha_0.7_PRIMARY_item_ids.npy

✓ All indexes saved to /kaggle/working/hnsw_indexes/
total 107M
-rw-r--r-- 1 root root  27M May 13 14:57 gallery_A_vision_only.bin
-rw-r--r-- 1 root root  96K May 13 14:57 gallery_A_vision_only_item_ids.npy
-rw-r--r-- 1 root root  27M May 13 14:57 gallery_B_alpha_0.7.bin
-rw-r--r-- 1 root root  96K Ma

In [20]:
import os, json, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

WORK_DIR = '/kaggle/working'
DEVICE = 'cuda:0'

# ---- Reload arrays from disk ----
all_meta       = pd.read_parquet(f'{WORK_DIR}/all_meta_ordered.parquet')
img_embeddings = np.load(f'{WORK_DIR}/img_embeddings.npy')
txt_embeddings = np.load(f'{WORK_DIR}/txt_embeddings.npy')

print(f"all_meta: {len(all_meta)} rows")
print(f"img: {img_embeddings.shape}, txt: {txt_embeddings.shape}")

# Recompute masks/ids
query_mask   = (all_meta.split == 'query').values
gallery_mask = (all_meta.split == 'gallery').values
q_ids = all_meta.loc[query_mask, 'item_id'].values
g_ids = all_meta.loc[gallery_mask, 'item_id'].values
q_txt = txt_embeddings[query_mask]
g_txt = txt_embeddings[gallery_mask]


# ---- Seed setter ----
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Seeded everything with {seed}")

In [21]:
from transformers import CLIPProcessor, CLIPModel
import hnswlib

CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
EMBED_DIM = 512

CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])


class InShopContrastiveDataset(Dataset):
    def __init__(self, train_meta, transform):
        self.transform = transform
        grouped = train_meta[train_meta.crop_path.notna()].groupby('item_id')['crop_path'].apply(list)
        self.items = grouped[grouped.apply(len) >= 2].to_dict()
        self.item_ids = list(self.items.keys())
    def __len__(self): return len(self.item_ids)
    def __getitem__(self, idx):
        views = self.items[self.item_ids[idx]]
        v1, v2 = random.sample(views, 2)
        return self.transform(Image.open(v1).convert("RGB")), self.transform(Image.open(v2).convert("RGB"))


def info_nce_loss(emb1, emb2, temperature=0.07):
    N = emb1.shape[0]
    logits = emb1 @ emb2.T / temperature
    labels = torch.arange(N, device=logits.device)
    return (nn.functional.cross_entropy(logits, labels) +
            nn.functional.cross_entropy(logits.T, labels)) / 2


def _to_tensor(x):
    if isinstance(x, torch.Tensor): return x
    for attr in ('image_embeds','pooler_output'):
        if hasattr(x, attr):
            t = getattr(x, attr)
            if isinstance(t, torch.Tensor): return t
    raise TypeError()


@torch.no_grad()
def embed_all_images(model, meta_df, batch_size=64):
    n = len(meta_df)
    out = np.zeros((n, EMBED_DIM), dtype=np.float32)
    for start in tqdm(range(0, n, batch_size), desc="Embed"):
        batch = meta_df.iloc[start:start + batch_size]
        imgs = [Image.open(p).convert("RGB") for p in batch['crop_path']]
        inputs = clip_processor(images=imgs, return_tensors="pt").to(DEVICE)
        feats = model.get_image_features(pixel_values=inputs['pixel_values'])
        feats = _to_tensor(feats)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        out[start:start + len(batch)] = feats.cpu().numpy()
    return out


def fuse(img_emb, txt_emb, alpha):
    v = alpha * img_emb + (1 - alpha) * txt_emb
    v = v / np.linalg.norm(v, axis=1, keepdims=True)
    return v.astype(np.float32)


def evaluate_retrieval(query_emb, query_ids, gallery_emb, gallery_ids, k_values=(5, 10, 15)):
    idx = hnswlib.Index(space='cosine', dim=gallery_emb.shape[1])
    idx.init_index(max_elements=len(gallery_emb), ef_construction=200, M=16)
    idx.add_items(gallery_emb, ids=np.arange(len(gallery_emb)))
    idx.set_ef(50)
    
    max_k = max(k_values)
    ids, _ = idx.knn_query(query_emb, k=max_k)
    
    metrics = {}
    for k in k_values:
        recalls, ndcgs, aps = [], [], []
        for i in range(len(query_emb)):
            true_id = query_ids[i]
            rel = (gallery_ids[ids[i][:k]] == true_id).astype(int)
            recalls.append(1.0 if rel.sum() > 0 else 0.0)
            dcg = np.sum(rel / np.log2(np.arange(2, k + 2)))
            ideal = np.sort(rel)[::-1]
            idcg = np.sum(ideal / np.log2(np.arange(2, k + 2)))
            ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
            if rel.sum() > 0:
                cum_hits = np.cumsum(rel)
                precision = cum_hits / np.arange(1, k + 1)
                aps.append(np.sum(precision * rel) / rel.sum())
            else:
                aps.append(0.0)
        metrics[f'Recall@{k}'] = float(np.mean(recalls))
        metrics[f'NDCG@{k}']   = float(np.mean(ndcgs))
        metrics[f'mAP@{k}']    = float(np.mean(aps))
    return metrics

In [22]:
def finetune_and_evaluate(seed, epochs=5, lr=1e-5, batch_size=64, alpha_eval=0.7):
    """
    One full fine-tuning + evaluation pass with the given seed.
    Returns metrics dict for Ablation C at α=alpha_eval (and α=1 vision-only).
    """
    set_seed(seed)
    
    # Fresh CLIP
    clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
    for p in clip_model.parameters():
        p.requires_grad = False
    for block in clip_model.vision_model.encoder.layers[-4:]:
        for p in block.parameters(): p.requires_grad = True
    for p in clip_model.visual_projection.parameters(): p.requires_grad = True
    for p in clip_model.vision_model.post_layernorm.parameters(): p.requires_grad = True
    
    # Dataset (uses seeded random.sample inside __getitem__)
    train_meta = all_meta[all_meta.split == 'train'].copy()
    train_dataset = InShopContrastiveDataset(train_meta, train_transform)
    
    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True, generator=g,
    )
    
    # Optimizer
    from torch.optim import AdamW
    from torch.optim.lr_scheduler import CosineAnnealingLR
    trainable = [p for p in clip_model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable, lr=lr, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs * len(train_loader))
    
    # Train
    clip_model.train()
    t0 = time.time()
    for epoch in range(epochs):
        epoch_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Seed {seed} | Epoch {epoch+1}/{epochs}")
        for img1, img2 in pbar:
            img1, img2 = img1.to(DEVICE), img2.to(DEVICE)
            e1 = clip_model.get_image_features(pixel_values=img1)
            e2 = clip_model.get_image_features(pixel_values=img2)
            e1 = _to_tensor(e1); e2 = _to_tensor(e2)
            e1 = e1 / e1.norm(dim=-1, keepdim=True)
            e2 = e2 / e2.norm(dim=-1, keepdim=True)
            loss = info_nce_loss(e1, e2)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        print(f"  Seed {seed} | Epoch {epoch+1}: avg loss = {epoch_loss/len(train_loader):.4f}")
    
    # Save checkpoint
    torch.save(clip_model.state_dict(), f'{WORK_DIR}/ft_clip_seed{seed}.pt')
    
    # Re-embed all images with this seed's model
    clip_model.eval()
    ft_img_emb = embed_all_images(clip_model, all_meta)
    np.save(f'{WORK_DIR}/ft_img_emb_seed{seed}.npy', ft_img_emb)
    
    # Evaluate
    q_ft_img = ft_img_emb[query_mask]
    g_ft_img = ft_img_emb[gallery_mask]
    
    metrics_vision = evaluate_retrieval(q_ft_img, q_ids, g_ft_img, g_ids)
    
    q_fused = fuse(q_ft_img, q_txt, alpha_eval)
    g_fused = fuse(g_ft_img, g_txt, alpha_eval)
    metrics_fused = evaluate_retrieval(q_fused, q_ids, g_fused, g_ids)
    
    # Cleanup
    del clip_model, ft_img_emb
    torch.cuda.empty_cache()
    
    print(f"  Seed {seed} done in {(time.time()-t0)/60:.1f} min")
    return {
        f'C_alpha_1.0_seed{seed}': metrics_vision,
        f'C_alpha_{alpha_eval}_seed{seed}': metrics_fused,
    }

In [23]:
SEEDS = [35,47, 58]   # ← replace with your team's actual roll numbers

all_seed_results = {}
for seed in SEEDS:
    print(f"\n{'='*70}")
    print(f"FINE-TUNING WITH SEED {seed}")
    print('='*70)
    seed_metrics = finetune_and_evaluate(seed)
    all_seed_results.update(seed_metrics)
    
    # Save after each seed so a crash doesn't wipe everything
    with open(f'{WORK_DIR}/results_seeded.json', 'w') as f:
        json.dump(all_seed_results, f, indent=2)

print("\n✓ All seeds done!")

FINE-TUNING WITH SEED 35
Seeded everything with 35
Loading widget...
Loading widget...
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Seed 35 | Epoch 1/5:   0%|          | 0/62 [00:00<?, ?it/s]
Loading widget...
Seed 35 | Epoch 1/5: 100%|██████████| 62/62 [01:18<00:00,  1.27s/it, loss=0.6589]
  Seed 35 | Epoch 1: avg loss = 1.1260
Seed 35 | Epoch 2/5: 100%|██████████| 62/62 [00:56<00:00,  1.10it/s, loss=0.5044]
  Seed 35 | Epoch 2: avg loss = 0.6295
Seed 35 | Epoch 3/5: 100%|██████████| 62/62 [00:50<00:00,  1.22it/s, loss=0.4999]
  Seed 35 | Epoch 3: avg loss = 0.5202
Seed 35 | Epoch 4/5: 100%|██████████| 62/62 [00:45<00:00,  1.36it/s, l

In [24]:
# Organize results: rows = seeds, cols = metric names
import pandas as pd

# Separate vision-only (α=1) and fused (α=0.7)
vision_metrics_by_seed = {}
fused_metrics_by_seed = {}

for key, metrics in all_seed_results.items():
    seed = int(key.split('seed')[-1])
    if 'alpha_1.0' in key:
        vision_metrics_by_seed[seed] = metrics
    else:
        fused_metrics_by_seed[seed] = metrics


def summarize(metrics_by_seed, label):
    df = pd.DataFrame(metrics_by_seed).T   # rows = seeds, cols = metrics
    mean = df.mean()
    std  = df.std()
    
    print(f"\n=== {label} (mean ± std over {len(df)} seeds) ===")
    summary_lines = []
    for col in df.columns:
        line = f"  {col}: {mean[col]:.4f} ± {std[col]:.4f}"
        print(line)
        summary_lines.append({'metric': col, 'mean': mean[col], 'std': std[col]})
    return pd.DataFrame(summary_lines)


vision_summary = summarize(vision_metrics_by_seed, "Ablation C — fine-tuned vision-only (α=1)")
fused_summary  = summarize(fused_metrics_by_seed, "Ablation C — fine-tuned + caption (α=0.7)")

# Save final summary
vision_summary.to_csv(f'{WORK_DIR}/ablation_C_vision_summary.csv', index=False)
fused_summary.to_csv(f'{WORK_DIR}/ablation_C_fused_summary.csv', index=False)

print(f"\n✓ Saved summary CSVs to {WORK_DIR}/")


=== Ablation C — fine-tuned vision-only (α=1) (mean ± std over 3 seeds) ===
  Recall@5: 0.7531 ± 0.0017
  NDCG@5: 0.6443 ± 0.0017
  mAP@5: 0.5967 ± 0.0017
  Recall@10: 0.8222 ± 0.0011
  NDCG@10: 0.6530 ± 0.0009
  mAP@10: 0.5728 ± 0.0013
  Recall@15: 0.8559 ± 0.0017
  NDCG@15: 0.6524 ± 0.0005
  mAP@15: 0.5529 ± 0.0010

=== Ablation C — fine-tuned + caption (α=0.7) (mean ± std over 3 seeds) ===
  Recall@5: 0.7777 ± 0.0015
  NDCG@5: 0.6745 ± 0.0009
  mAP@5: 0.6288 ± 0.0008
  Recall@10: 0.8404 ± 0.0013
  NDCG@10: 0.6811 ± 0.0009
  mAP@10: 0.6045 ± 0.0008
  Recall@15: 0.8718 ± 0.0010
  NDCG@15: 0.6798 ± 0.0009
  mAP@15: 0.5846 ± 0.0008

✓ Saved summary CSVs to /kaggle/working/